# 100 — Export hierarchically integrated virtual T1 survey to SEG-Y (SAFE, v5)

Exports the authoritative sparse shot gathers and an optional rectangular
receiver grid. Because notebook 99 now permits only one authoritative trace per
receiver position, the regularized grid is keyed by receiver coordinate rather
than by receiver family.

## 1. Configuration

In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from obspy import read, Stream, Trace
from obspy.io.segy.segy import SEGYTraceHeader, SEGYBinaryFileHeader
from obspy.core import AttribDict

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
INPUT_ROOT = PROJECT_ROOT / '99_virtual_T1_shot_gathers'
OUT_ROOT = PROJECT_ROOT / '100_virtual_T1_SEGY'
PER_SHOT_ROOT = OUT_ROOT / 'per_shot_segy'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
PER_SHOT_ROOT.mkdir(parents=True, exist_ok=True)

SHOT_MANIFEST_PATH = INPUT_ROOT / '99_virtual_T1_shot_manifest.csv'
TRACE_MANIFEST_PATH = INPUT_ROOT / '99_virtual_T1_trace_manifest.csv'

COMPONENT = 'Z'
DATA_ENCODING = 5  # IEEE 32-bit float
BYTEORDER = '>'
WRITE_PER_SHOT_SEGY = True
WRITE_SPARSE_ALL_SHOTS = True
WRITE_REGULARIZED_ALL_SHOTS = True
REQUIRE_GEODE_TRACES = True

# Export-time geometry safeguards. Notebook 99 remains authoritative for
# geometry, but notebook 100 refuses to write clearly invalid products.
DUPLICATE_RECEIVER_TOLERANCE_M = 0.001
STREAMER_NEAREST_OFFSET_M = 30.0 * 0.3048
STREAMER_FURTHEST_OFFSET_M = 145.0 * 0.3048
STREAMER_GEOMETRY_TOLERANCE_M = 0.25
FAIL_ON_GEOMETRY_ERROR = True
FAIL_ON_UNRESOLVED_NODAL_TIMING = True

# Coordinates are stored as integer millimetres with scalar -1000.
COORDINATE_SCALAR = -1000
COORDINATE_MULTIPLIER = 1000

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Input:', INPUT_ROOT)
print('Output:', OUT_ROOT)

Input: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers
Output: /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY


## 2. Load ordered shot and trace catalogs

In [2]:
for path in [SHOT_MANIFEST_PATH, TRACE_MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing notebook-99 output: {path}')

shots = pd.read_csv(SHOT_MANIFEST_PATH, low_memory=False)
traces = pd.read_csv(TRACE_MANIFEST_PATH, low_memory=False)

shots = shots.loc[
    shots.status.eq('written')
    & shots.component.astype(str).str.upper().eq(COMPONENT)
].copy()

shots = shots.sort_values(
    ['source_x_m', 'virtual_shot_number'],
    kind='stable',
).reset_index(drop=True)

traces['source_x_m'] = pd.to_numeric(
    traces.source_x_m, errors='coerce'
)
traces['receiver_x_m'] = pd.to_numeric(
    traces.receiver_x_m, errors='coerce'
)



unresolved_nodal_timing = shots.loc[
    shots.get('nodal_only_shot', False).astype(str).str.lower().isin(
        ['true', '1', 'yes']
    )
    & ~shots.get(
        'shot_timing_status',
        '',
    ).astype(str).isin([
        'accepted',
        'accepted_manual_override',
    ])
].copy()

if len(unresolved_nodal_timing):
    display(unresolved_nodal_timing[[
        'virtual_shot_number',
        'virtual_shot_id',
        'source_x_m',
        'shot_timing_method',
        'shot_timing_status',
    ]].head(50))

    if FAIL_ON_UNRESOLVED_NODAL_TIMING:
        raise RuntimeError(
            f'{len(unresolved_nodal_timing)} nodal-only shots do not '
            'have an accepted physical time origin. Review '
            '99_virtual_T1_shot_time_origin_qc.csv or enable a manual '
            'override before SEG-Y export.'
        )

print('Shots to export:', len(shots))
print('Observed traces:', len(traces))
receiver_family_counts = traces.receiver_family.value_counts(dropna=False)
display(receiver_family_counts.rename_axis('receiver_family').reset_index(name='n_traces'))
if REQUIRE_GEODE_TRACES and not traces.receiver_family.astype(str).eq('geode').any():
    raise RuntimeError(
        'No Geode traces are present in the notebook-99 trace manifest. '
        'Rerun updated notebooks 98 and 99 before exporting SEG-Y.'
    )

,virtual_shot_number,virtual_shot_id,source_x_m,shot_timing_method,shot_timing_status
1,2,T1_VSHOT_0002_x00036.0m,36.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks
36,37,T1_VSHOT_0037_x00106.0m,106.0,nodal_airwave_backprojection_qc,inconsistent_receiver_residuals
41,42,T1_VSHOT_0042_x00109.0m,109.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks
43,44,T1_VSHOT_0044_x00110.0m,110.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks
46,47,T1_VSHOT_0047_x00112.0m,112.0,nodal_airwave_backprojection_qc,inconsistent_receiver_residuals
48,49,T1_VSHOT_0049_x00113.0m,113.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks
53,54,T1_VSHOT_0054_x00116.0m,116.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks
56,57,T1_VSHOT_0057_x00118.0m,118.0,nodal_airwave_backprojection_qc,biased_receiver_residuals
63,64,T1_VSHOT_0064_x00122.0m,122.0,nodal_airwave_backprojection_qc,inconsistent_receiver_residuals
66,67,T1_VSHOT_0067_x00124.0m,124.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks


RuntimeError: 20 nodal-only shots do not have an accepted physical time origin. Review 99_virtual_T1_shot_time_origin_qc.csv or enable a manual override before SEG-Y export.

In [3]:
import pandas as pd
from pathlib import Path

qc_path = Path(
    "/Volumes/tachyon/LBSSP_DATA/"
    "99_virtual_T1_shot_gathers/"
    "99_virtual_T1_shot_time_origin_qc.csv"
)

qc = pd.read_csv(qc_path)

nodal = qc.loc[qc["nodal_only_shot"].astype(bool)].copy()

display(
    nodal.groupby(
        ["timing_status", "fit_status"],
        dropna=False,
    )
    .size()
    .reset_index(name="n_shots")
    .sort_values("n_shots", ascending=False)
)

display(
    nodal[
        [
            "virtual_shot_number",
            "virtual_shot_id",
            "source_x_m",
            "timing_method",
            "timing_status",
            "fit_status",
            "estimated_shot_time_from_file_start_s",
            "airwave_velocity_mps",
            "airwave_score",
            "airwave_peak_ratio",
            "n_receivers_used",
            "n_supporting_receivers",
            "receiver_residual_median_s",
            "receiver_residual_mad_s",
        ]
    ].sort_values("source_x_m")
)

,timing_status,fit_status,n_shots
1,ambiguous_multiple_airwave_peaks,ambiguous_multiple_airwave_peaks,12
3,inconsistent_receiver_residuals,inconsistent_receiver_residuals,5
0,accepted,accepted,3
2,biased_receiver_residuals,biased_receiver_residuals,3


,virtual_shot_number,virtual_shot_id,source_x_m,timing_method,timing_status,fit_status,estimated_shot_time_from_file_start_s,airwave_velocity_mps,airwave_score,airwave_peak_ratio,n_receivers_used,n_supporting_receivers,receiver_residual_median_s,receiver_residual_mad_s
0,1,T1_VSHOT_0001_x00010.0m,10.0,nodal_airwave_backprojection,accepted,accepted,0.117,369.0,3.715223,1.202987,32,31,0.003415,0.010953
1,2,T1_VSHOT_0002_x00036.0m,36.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks,ambiguous_multiple_airwave_peaks,-0.073,370.0,10.204837,1.004465,34,33,-0.000189,0.013624
33,34,T1_VSHOT_0034_x00104.0m,104.0,nodal_airwave_backprojection,accepted,accepted,0.229,321.0,41.680742,1.341232,20,20,0.001581,0.014417
36,37,T1_VSHOT_0037_x00106.0m,106.0,nodal_airwave_backprojection_qc,inconsistent_receiver_residuals,inconsistent_receiver_residuals,0.176,322.0,28.504284,1.311698,20,20,-0.003922,0.022663
41,42,T1_VSHOT_0042_x00109.0m,109.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks,ambiguous_multiple_airwave_peaks,0.296,368.0,34.887143,1.144488,19,19,0.004011,0.014745
43,44,T1_VSHOT_0044_x00110.0m,110.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks,ambiguous_multiple_airwave_peaks,0.311,366.0,29.320221,1.074273,18,18,-0.003757,0.011723
46,47,T1_VSHOT_0047_x00112.0m,112.0,nodal_airwave_backprojection_qc,inconsistent_receiver_residuals,inconsistent_receiver_residuals,0.255,322.0,56.362443,1.392623,18,18,0.002879,0.023169
48,49,T1_VSHOT_0049_x00113.0m,113.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks,ambiguous_multiple_airwave_peaks,0.245,321.0,40.318700,1.051787,18,18,0.006850,0.018761
53,54,T1_VSHOT_0054_x00116.0m,116.0,nodal_airwave_backprojection_qc,ambiguous_multiple_airwave_peaks,ambiguous_multiple_airwave_peaks,0.271,370.0,31.552693,1.033495,17,17,0.002270,0.017591
56,57,T1_VSHOT_0057_x00118.0m,118.0,nodal_airwave_backprojection_qc,biased_receiver_residuals,biased_receiver_residuals,0.271,320.0,40.012868,1.467142,16,16,-0.015453,0.006371


In [ ]:
def geometry_validation_rows(shots, traces):
    rows = []
    for shot in shots.itertuples(index=False):
        frame = traces.loc[
            traces.virtual_shot_number.eq(shot.virtual_shot_number)
        ].copy()
        if not len(frame):
            rows.append({
                'virtual_shot_number': int(shot.virtual_shot_number),
                'source_x_m': float(shot.source_x_m),
                'status': 'no_traces',
                'message': 'No trace-manifest rows.',
            })
            continue

        x = pd.to_numeric(frame.receiver_x_m, errors='coerce').to_numpy()
        finite = x[np.isfinite(x)]
        rounded = np.round(
            finite / DUPLICATE_RECEIVER_TOLERANCE_M
        ).astype(np.int64)
        duplicate_count = int(len(rounded) - len(np.unique(rounded)))

        messages = []
        if duplicate_count:
            messages.append(
                f'{duplicate_count} duplicate receiver positions within '
                f'{DUPLICATE_RECEIVER_TOLERANCE_M} m'
            )

        streamer = frame.loc[
            frame.integration_role.astype(str).eq('T1_Streamer')
        ].copy()
        for row in streamer.itertuples(index=False):
            offset = float(shot.source_x_m) - float(row.receiver_x_m)
            if not (
                STREAMER_NEAREST_OFFSET_M - STREAMER_GEOMETRY_TOLERANCE_M
                <= offset
                <= STREAMER_FURTHEST_OFFSET_M + STREAMER_GEOMETRY_TOLERANCE_M
            ):
                messages.append(
                    'streamer receiver outside expected source-relative '
                    f'range: receiver={row.receiver_x_m}, offset={offset}'
                )

        rows.append({
            'virtual_shot_number': int(shot.virtual_shot_number),
            'source_x_m': float(shot.source_x_m),
            'n_traces': len(frame),
            'receiver_x_min_m': float(np.nanmin(finite)),
            'receiver_x_max_m': float(np.nanmax(finite)),
            'duplicate_position_count': duplicate_count,
            'status': 'error' if messages else 'ok',
            'message': ' | '.join(messages),
        })
    return pd.DataFrame(rows)


geometry_validation = geometry_validation_rows(shots, traces)
display(geometry_validation.loc[geometry_validation.status.ne('ok')].head(50))

if FAIL_ON_GEOMETRY_ERROR and geometry_validation.status.eq('error').any():
    raise RuntimeError(
        'Geometry validation failed. Review '
        '100_virtual_T1_geometry_validation.csv after temporarily setting '
        'FAIL_ON_GEOMETRY_ERROR=False, or correct notebook 99 outputs.'
    )

## 3. SEG-Y header helpers

In [ ]:
def coordinate_integer(value_m):
    return int(round(float(value_m) * COORDINATE_MULTIPLIER))


def attach_segy_header(
    trace,
    *,
    global_trace_number,
    shot_number,
    trace_number_within_shot,
    source_x_m,
    receiver_x_m,
    is_live,
    relative_start_s,
):
    trace.stats.segy = AttribDict()
    header = SEGYTraceHeader()

    header.trace_sequence_number_within_line = int(
        global_trace_number
    )
    header.trace_sequence_number_within_segy_file = int(
        global_trace_number
    )
    header.original_field_record_number = int(shot_number)
    header.trace_number_within_the_original_field_record = int(
        trace_number_within_shot
    )
    header.energy_source_point_number = int(shot_number)

    header.scalar_to_be_applied_to_all_coordinates = (
        COORDINATE_SCALAR
    )
    header.coordinate_units = 1
    header.source_coordinate_x = coordinate_integer(source_x_m)
    header.group_coordinate_x = coordinate_integer(receiver_x_m)
    header.distance_from_center_of_the_source_point_to_the_center_of_the_receiver_group = (
        coordinate_integer(receiver_x_m - source_x_m)
    )

    # 1 = seismic data, 2 = dead trace.
    header.trace_identification_code = 1 if is_live else 2
    delay_ms = int(round(float(relative_start_s) * 1000.0))
    if not -32768 <= delay_ms <= 32767:
        raise ValueError(
            f'SEG-Y delay_recording_time overflow: {delay_ms} ms'
        )
    header.delay_recording_time = delay_ms

    trace.stats.segy.trace_header = header
    return trace


def write_segy(
    stream,
    path,
    *,
    traces_per_ensemble=0,
):
    """
    Write SEG-Y while explicitly setting the binary-header number of data
    traces per ensemble.

    ObsPy otherwise uses the total number of traces in the Stream. For an
    all-shot file that can exceed the signed 16-bit SEG-Y field, even though
    the actual number of traces in each shot ensemble is much smaller.
    """
    if not len(stream):
        return

    traces_per_ensemble = int(traces_per_ensemble)
    if not 0 <= traces_per_ensemble <= 32767:
        raise ValueError(
            'traces_per_ensemble must fit the signed 16-bit SEG-Y '
            f'header field; got {traces_per_ensemble}.'
        )

    stream.stats = AttribDict()
    stream.stats.binary_file_header = SEGYBinaryFileHeader()
    stream.stats.binary_file_header.number_of_data_traces_per_ensemble = (
        traces_per_ensemble
    )
    stream.stats.binary_file_header.number_of_auxiliary_traces_per_ensemble = 0

    stream.write(
        str(path),
        format='SEGY',
        data_encoding=DATA_ENCODING,
        byteorder=BYTEORDER,
    )

## 4. Read shot gathers and construct sparse SEG-Y stream

In [ ]:
sparse_stream = Stream()
sparse_catalog_rows = []
global_trace_number = 0
per_shot_streams = {}

for shot in shots.itertuples(index=False):
    path = Path(str(shot.mseed_path))
    if not path.exists():
        print('WARNING: missing MiniSEED:', path)
        continue

    stream = read(str(path))
    shot_trace_catalog = traces.loc[
        traces.virtual_shot_number.eq(shot.virtual_shot_number)
    ].copy()

    lookup = {
        (
            str(row.station),
            str(row.channel),
        ): row
        for row in shot_trace_catalog.itertuples(index=False)
    }

    prepared = []

    for trace in stream:
        key = (str(trace.stats.station), str(trace.stats.channel))
        row = lookup.get(key)
        if row is None:
            continue
        prepared.append((row, trace.copy()))

    prepared.sort(
        key=lambda item: (
            float(item[0].receiver_x_m),
            str(item[0].receiver_family),
            str(item[0].station),
        )
    )

    shot_stream = Stream()

    for trace_number, (row, trace) in enumerate(prepared, start=1):
        global_trace_number += 1
        trace.data = np.asarray(trace.data, dtype=np.float32)
        attach_segy_header(
            trace,
            global_trace_number=global_trace_number,
            shot_number=int(shot.virtual_shot_number),
            trace_number_within_shot=trace_number,
            source_x_m=float(shot.source_x_m),
            receiver_x_m=float(row.receiver_x_m),
            is_live=True,
        relative_start_s=float(getattr(row, 'relative_start_s', getattr(shot, 'relative_start_s', 0.0))),
        )
        shot_stream += trace
        sparse_stream += trace.copy()

        sparse_catalog_rows.append({
            'global_trace_number': global_trace_number,
            'virtual_shot_number': int(shot.virtual_shot_number),
            'virtual_shot_id': shot.virtual_shot_id,
            'source_x_m': float(shot.source_x_m),
            'trace_number_within_shot': trace_number,
            'receiver_family': row.receiver_family,
            'receiver_x_m': float(row.receiver_x_m),
            'station': row.station,
            'channel': row.channel,
            'trace_identification_code': 1,
            'is_live': True,
            'relative_start_s': float(
                getattr(
                    row,
                    'relative_start_s',
                    getattr(shot, 'relative_start_s', 0.0),
                )
            ),
            'shot_timing_method': getattr(
                shot,
                'shot_timing_method',
                None,
            ),
            'shot_timing_status': getattr(
                shot,
                'shot_timing_status',
                None,
            ),
        })

    per_shot_streams[int(shot.virtual_shot_number)] = shot_stream

print('Sparse SEG-Y traces:', len(sparse_stream))

## 5. Write sparse all-shot and per-shot SEG-Y

In [ ]:
SPARSE_PATH = OUT_ROOT / f'T1_virtual_sparse_{COMPONENT}.segy'

if WRITE_SPARSE_ALL_SHOTS:
    write_segy(
        sparse_stream,
        SPARSE_PATH,
        traces_per_ensemble=0,
    )
    print('Wrote:', SPARSE_PATH)

if WRITE_PER_SHOT_SEGY:
    for shot in shots.itertuples(index=False):
        shot_stream = per_shot_streams.get(
            int(shot.virtual_shot_number),
            Stream(),
        )
        if not len(shot_stream):
            continue
        filename = (
            f'T1_VIRTUAL_SHOT_{int(shot.virtual_shot_number):04d}'
            f'_x{float(shot.source_x_m):07.1f}m_{COMPONENT}.segy'
        )
        write_segy(
            shot_stream,
            PER_SHOT_ROOT / filename,
            traces_per_ensemble=len(shot_stream),
        )

    print('Per-shot SEG-Y directory:', PER_SHOT_ROOT)

## 6. Build regularized receiver grid

In [ ]:
# One master receiver coordinate per position. Receiver family can vary between
# shots depending on which highest-priority product supplied that location.
master_receivers = (
    traces[['receiver_x_m']]
    .assign(receiver_x_key=lambda frame: frame.receiver_x_m.round(6))
    .drop_duplicates('receiver_x_key')
    .sort_values('receiver_x_m', kind='stable')
    .reset_index(drop=True)
)
master_receivers.insert(0, 'master_receiver_number', np.arange(1, len(master_receivers) + 1, dtype=int))
master_receivers['channel'] = 'GHZ'

print('Master receiver positions:', len(master_receivers))
display(master_receivers.head(30))

## 7. Write regularized all-shot SEG-Y with dead traces

In [ ]:
regularized_stream = Stream()
regularized_catalog_rows = []
global_trace_number = 0

if WRITE_REGULARIZED_ALL_SHOTS:
    for shot in shots.itertuples(index=False):
        shot_stream = per_shot_streams.get(int(shot.virtual_shot_number), Stream())
        shot_trace_catalog = traces.loc[
            traces.virtual_shot_number.eq(shot.virtual_shot_number)
        ].copy()

        observed_lookup = {}
        for row in shot_trace_catalog.itertuples(index=False):
            candidates = [
                tr for tr in shot_stream
                if str(tr.stats.station) == str(row.station)
                and str(tr.stats.channel) == str(row.channel)
            ]
            if candidates:
                observed_lookup[round(float(row.receiver_x_m), 6)] = (row, candidates[0].copy())

        if not len(shot_stream):
            raise RuntimeError(f'No template trace available for shot {shot.virtual_shot_number}')
        template = shot_stream[0]

        for trace_number, receiver in enumerate(master_receivers.itertuples(index=False), start=1):
            key = round(float(receiver.receiver_x_m), 6)
            is_live = key in observed_lookup
            if is_live:
                row, trace = observed_lookup[key]
                receiver_family = row.receiver_family
                integration_role = getattr(row, 'integration_role', receiver_family)
            else:
                trace = Trace(data=np.zeros(template.stats.npts, dtype=np.float32))
                trace.stats.sampling_rate = template.stats.sampling_rate
                trace.stats.starttime = template.stats.starttime
                trace.stats.network = 'VT'
                trace.stats.station = f'D{int(receiver.master_receiver_number):04d}'[:5]
                trace.stats.location = 'DD'
                trace.stats.channel = 'GHZ'
                receiver_family = 'dead'
                integration_role = 'dead'

            global_trace_number += 1
            trace.data = np.asarray(trace.data, dtype=np.float32)
            attach_segy_header(
                trace,
                global_trace_number=global_trace_number,
                shot_number=int(shot.virtual_shot_number),
                trace_number_within_shot=trace_number,
                source_x_m=float(shot.source_x_m),
                receiver_x_m=float(receiver.receiver_x_m),
                is_live=is_live,
            relative_start_s=float(getattr(shot, 'relative_start_s', 0.0)),
            )
            regularized_stream += trace
            regularized_catalog_rows.append({
                'global_trace_number': global_trace_number,
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'trace_number_within_shot': trace_number,
                'master_receiver_number': int(receiver.master_receiver_number),
                'receiver_family': receiver_family,
                'integration_role': integration_role,
                'receiver_x_m': float(receiver.receiver_x_m),
                'channel': trace.stats.channel,
                'trace_identification_code': 1 if is_live else 2,
                'is_live': is_live,
            })

    REGULARIZED_PATH = OUT_ROOT / f'T1_virtual_regularized_{COMPONENT}.segy'
    write_segy(
        regularized_stream,
        REGULARIZED_PATH,
        traces_per_ensemble=len(master_receivers),
    )
    print('Wrote:', REGULARIZED_PATH)
    print('Regularized traces:', len(regularized_stream))

## 8. Export SEG-Y catalogs and summary

In [ ]:
sparse_catalog = pd.DataFrame(sparse_catalog_rows)
regularized_catalog = pd.DataFrame(regularized_catalog_rows)

OUTPUTS = {
    'sparse_catalog': OUT_ROOT / '100_virtual_T1_sparse_trace_catalog.csv',
    'master_receivers': OUT_ROOT / '100_virtual_T1_master_receiver_grid.csv',
    'regularized_catalog': OUT_ROOT / '100_virtual_T1_regularized_trace_catalog.csv',
    'summary': OUT_ROOT / '100_virtual_T1_SEGY_summary.csv',
    'geometry_validation': OUT_ROOT / '100_virtual_T1_geometry_validation.csv',
}

sparse_catalog.to_csv(OUTPUTS['sparse_catalog'], index=False)
master_receivers.to_csv(OUTPUTS['master_receivers'], index=False)
regularized_catalog.to_csv(
    OUTPUTS['regularized_catalog'], index=False
)
geometry_validation.to_csv(OUTPUTS['geometry_validation'], index=False)

summary = pd.DataFrame([
    ('shots_exported', len(shots)),
    ('sparse_live_traces', len(sparse_catalog)),
    ('sparse_nodal_traces', int(sparse_catalog.receiver_family.eq('nodal').sum()) if len(sparse_catalog) else 0),
    ('sparse_geode_traces', int(sparse_catalog.receiver_family.eq('geode').sum()) if len(sparse_catalog) else 0),
    ('master_receiver_positions', len(master_receivers)),
    ('binary_header_traces_per_ensemble', len(master_receivers)),
    ('regularized_total_traces', len(regularized_catalog)),
    ('regularized_live_traces', int(
        regularized_catalog.is_live.sum()
    ) if len(regularized_catalog) else 0),
    ('regularized_dead_traces', int(
        (~regularized_catalog.is_live).sum()
    ) if len(regularized_catalog) else 0),
], columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for name, path in OUTPUTS.items():
    print(f'  {name:20s} {path}')

## 9. Interpretation

Use `T1_virtual_sparse_Z.segy` as the authoritative observed survey.

Use `T1_virtual_regularized_Z.segy` only when downstream software requires a
rectangular source-receiver matrix. A zero-valued trace with SEG-Y
`trace_identification_code = 2` is missing/dead data, not an observed zero
amplitude.

The trace catalogs preserve receiver family and live/dead status explicitly.